In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import chess
import numpy as np
import pandas as pd
import math

In [18]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_default_device('cpu')

In [19]:
torch.manual_seed(7)
generator = torch.Generator(7)

In [ ]:
def getInput(fen):
    board = chess.Board(fen)
    x = np.zeros((787, ), dtype=np.float32)
    whiteBishop = 0
    blackBishop = 0
    phase = 0
    for sq in range(64):
        p = board.piece_at(sq)
        if(p != None):
            match p.piece_type:
                case(chess.KING):
                    x[sq + (384 if p.color == chess.BLACK else 0)] = 1
                case(chess.QUEEN):
                    x[sq + 64 + (384 if p.color == chess.BLACK else 0)] = 1
                    phase += 4
                    if p.color == chess.WHITE:
                        x[775] += 1
                    else:
                        x[780] += 1
                case(chess.ROOK):
                    x[sq + 128 + (384 if p.color == chess.BLACK else 0)] = 1
                    phase += 2
                    if p.color == chess.WHITE:
                        x[776] += 1
                    else:
                        x[781] += 1
                case(chess.BISHOP):
                    x[sq + 192 + (384 if p.color == chess.BLACK else 0)] = 1
                    phase += 1
                    if p.color == chess.WHITE:
                        whiteBishop += 1
                        x[777] += 1
                    else:
                        blackBishop += 1
                        x[782] += 1
                case(chess.KNIGHT):
                    x[sq + 256 + (384 if p.color == chess.BLACK else 0)] = 1
                    phase += 1
                    if p.color == chess.WHITE:
                        x[778] += 1
                    else:
                        x[783] += 1
                case(chess.PAWN):
                    x[sq + 320 + (384 if p.color == chess.BLACK else 0)] = 1
                    if p.color == chess.WHITE:
                        x[779] += 1
                    else:
                        x[784] += 1
    x[768] = board.has_kingside_castling_rights(chess.WHITE)
    x[769] = board.has_queenside_castling_rights(chess.WHITE)
    x[770] = board.has_kingside_castling_rights(chess.BLACK)
    x[771] = board.has_queenside_castling_rights(chess.BLACK)
    x[772] = (1 if board.turn == chess.WHITE else 0)
    x[773] = (1 if whiteBishop == 2 else 0)
    x[774] = (1 if blackBishop == 2 else 0)
    x[775] /= 1.0
    x[776] /= 2.0 
    x[777] /= 2.0  
    x[778] /= 2.0   
    x[779] /= 8.0  
    x[780] /= 1.0
    x[781] /= 2.0
    x[782] /= 2.0
    x[783] /= 2.0
    x[784] /= 8.0
    x[785] = phase / 24.0
    whiteMaterial = 9 * x[775] + 5 * x[776] + 3 * x[777] + 3 * x[778] + 1 * x[779]
    blackMaterial = 9 * x[780] + 5 * x[781] + 3 * x[782] + 3 * x[783] + 1 * x[784]
    x[786] = (whiteMaterial - blackMaterial)/39.0
    
    return x


In [21]:
class ChessDataset(Dataset):
    def __init__(self, file):
        super().__init__()
        self.df = pd.read_csv(file)
        
    def __len__(self):
        return self.df.shape[0]
        
    def __getitem__(self, index):
        fen = self.df.iloc[index, 0]
        score = self.df.iloc[index, 1]
            
        return torch.from_numpy(getInput(fen)).float(), torch.tensor(score, dtype=torch.float32)
    
file = "FilteredEvals.csv"
dataset = ChessDataset(file)
train_data, test_data = random_split(dataset=dataset, lengths=[0.8, 0.2])

In [22]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(787, 768)
        self.fc2 = nn.Linear(768, 432)
        self.fc3 = nn.Linear(432, 128)
        self.fc4 = nn.Linear(128, 1)
    
    def forward(self, input):
        f1 = self.fc1(input)
        f2 = F.relu(f1)
        f3 = self.fc2(f2)
        f4 = F.relu(f3)
        f5 = self.fc3(f4)
        f6 = F.relu(f5)
        out = self.fc4(f6)
        return out 

In [23]:
epochs = 20
batch_size = 1024
dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=True)
n_iterations = math.ceil(len(train_data) / batch_size)

In [24]:
model = Network().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.SmoothL1Loss()

best_mae = float("inf")

for epoch in range(epochs):


    model.train()

    running_loss = 0

    for i, (input, label) in enumerate(dataloader):

        input = input.to(device)
        label = label.unsqueeze(1).to(device)

        optimizer.zero_grad()

        output = model(input)

        loss = criterion(output / 100, label / 100)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(dataloader)


    model.eval()

    total_mae = 0

    with torch.no_grad():

        for input, label in test_loader:

            input = input.to(device)
            label = label.unsqueeze(1).to(device)

            pred = model(input)

            total_mae += torch.abs(pred - label).sum().item()

    val_mae = total_mae / len(test_data)

    print(
        f"Epoch {epoch+1:2d} | "
        f"Train Loss = {train_loss:.4f} | "
        f"Validation MAE = {val_mae:.2f} cp"
    )

    if val_mae < best_mae:

        best_mae = val_mae

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_mae": val_mae,
        }, "best_nnue_checkpoint.pt")

        print("Saved new best model!")


torch.save(model.state_dict(), "final_nnue_weights.pt")

print("Training finished.")

Epoch  1 | Train Loss = 2.8393 | Validation MAE = 305.05 cp
Saved new best model!
Epoch  2 | Train Loss = 2.3747 | Validation MAE = 270.95 cp
Saved new best model!
Epoch  3 | Train Loss = 1.9815 | Validation MAE = 252.58 cp
Saved new best model!
Epoch  4 | Train Loss = 1.7379 | Validation MAE = 242.28 cp
Saved new best model!
Epoch  5 | Train Loss = 1.5880 | Validation MAE = 236.80 cp
Saved new best model!
Epoch  6 | Train Loss = 1.4860 | Validation MAE = 231.32 cp
Saved new best model!
Epoch  7 | Train Loss = 1.4040 | Validation MAE = 228.00 cp
Saved new best model!
Epoch  8 | Train Loss = 1.3363 | Validation MAE = 225.92 cp
Saved new best model!
Epoch  9 | Train Loss = 1.2849 | Validation MAE = 223.35 cp
Saved new best model!
Epoch 10 | Train Loss = 1.2377 | Validation MAE = 221.74 cp
Saved new best model!
Epoch 11 | Train Loss = 1.1962 | Validation MAE = 220.13 cp
Saved new best model!
Epoch 12 | Train Loss = 1.1589 | Validation MAE = 219.04 cp
Saved new best model!
Epoch 13 | Train

In [25]:
model.eval()

with torch.no_grad():

    input, label = next(iter(test_loader))

    input = input.to(device)

    pred = model(input).cpu()

    for i in range(20):
        print(
            f"Target: {label[i].item():7.1f}   "
            f"Prediction: {pred[i].item():7.1f}"
        )

Target:   247.0   Prediction:   112.6
Target:  -509.0   Prediction:  -559.7
Target:   160.0   Prediction:   117.6
Target:   614.0   Prediction:   444.9
Target:   409.0   Prediction:  -463.7
Target:  -242.0   Prediction:    61.2
Target:    23.0   Prediction:    -7.8
Target:  -239.0   Prediction:  -292.1
Target:   136.0   Prediction:   -50.4
Target:  -567.0   Prediction:  -291.9
Target:    46.0   Prediction:     8.9
Target:    -8.0   Prediction:  -280.3
Target:   116.0   Prediction:   228.6
Target:   482.0   Prediction:   471.4
Target:   226.0   Prediction:   144.1
Target:   314.0   Prediction:  -324.9
Target:  -597.0   Prediction:  -506.0
Target:   136.0   Prediction:   187.9
Target:  -230.0   Prediction:   -93.4
Target:   191.0   Prediction:    60.3


In [26]:
from scipy.stats import pearsonr

predictions = []
targets = []

model.eval()

with torch.no_grad():
    for x, y in test_loader:
        pred = model(x.to(device)).cpu().flatten()

        predictions.extend(pred.tolist())
        targets.extend(y.tolist())

corr, _ = pearsonr(predictions, targets)

print(corr)

0.6312238865526363


In [27]:
state = model.state_dict()

for name, tensor in state.items():
    np.save(name + ".npy", tensor.cpu().numpy())

In [28]:
state = model.state_dict()

for name, tensor in state.items():
    np.savetxt(
        name + ".txt",
        tensor.cpu().numpy().reshape(-1),
        fmt="%.10f"
    )